In [1]:
print("Hello, Colab!")

Hello, Colab!


In [4]:
import torch
print(torch.cuda.is_available())


True


In [2]:
!pip -q install transformers accelerate bitsandbytes

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


TypeError: Qwen2ForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [6]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [9]:
def ask_llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        top_p=0.9
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(ask_llm("Write one interview question for a junior backend developer."))

Write one interview question for a junior backend developer. How can you approach debugging and troubleshooting issues in a large-scale, distributed system with multiple microservices? As an AI language model, I don't have personal experience or skills to write the answer. However, here is an example of how a potential question could look like:

Interviewer: Can you describe your approach when encountering technical challenges while working on a large-scale, distributed system that consists of several microservices?

The candidate's response should cover their understanding of best practices for handling errors and failures in a distributed environment, such as logging, monitoring, and implementing retry policies. They should also discuss any tools


In [10]:
def generate_question(role, level, topic, difficulty):
    prompt = f"""
You are a professional mock interviewer.

Generate exactly ONE interview question.

Rules:
- Role: {role}
- Level: {level}
- Topic: {topic}
- Difficulty: {difficulty}
- Ask only one question
- Do not give the answer
- Keep it realistic and professional
- Output only the question text

Question:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response

In [11]:
q = generate_question(
    role="Backend Developer",
    level="Junior",
    topic="REST API",
    difficulty="Easy"
)

print(q)


You are a professional mock interviewer.

Generate exactly ONE interview question.

Rules:
- Role: Backend Developer
- Level: Junior
- Topic: REST API
- Difficulty: Easy
- Ask only one question
- Do not give the answer
- Keep it realistic and professional
- Output only the question text

Question:
What is the main difference between GET, POST, PUT, and DELETE HTTP methods in terms of their purpose and usage in building RESTful APIs? Sure, here's your question:

### Question:

"What is the main difference between `GET`, `POST`, `PUT`, and `DELETE` HTTP methods in terms of their purpose and usage in building RESTful APIs?"

This question focuses on the core differences


In [13]:
def generate_question(role, level, topic, difficulty):
    prompt = f"""
You are a professional mock interviewer.

Generate exactly ONE interview question.

Rules:
- Role: {role}
- Level: {level}
- Topic: {topic}
- Difficulty: {difficulty}
- Ask only one question
- Do not give the answer
- Keep it realistic and professional
- Output only the question text

Question:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    full_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Keep only generated part after "Question:"
    if "Question:" in full_text:
        result = full_text.split("Question:")[-1].strip()
    else:
        result = full_text.strip()

    return result

In [14]:
q = generate_question(
    role="Backend Developer",
    level="Junior",
    topic="REST API",
    difficulty="Easy"
)

print(q)

What is the difference between GET, POST, PUT, and DELETE methods in RESTful web services? To help you understand, please provide an example for each method.


In [35]:
def evaluate_answer_json(question, answer):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert technical interviewer. "
                "Return ONLY valid JSON. "
                "Do not use markdown. "
                "Do not use code fences. "
                "All values must be valid JSON. "
                "The improved_answer must be a single string."
            )
        },
        {
            "role": "user",
            "content": f"""
Question: {question}

Candidate Answer:
{answer}

Evaluate the answer on:
- technical_accuracy (0 to 5)
- clarity (0 to 5)
- depth (0 to 5)

Return exactly this JSON structure:
{{
  "technical_accuracy": 0,
  "clarity": 0,
  "depth": 0,
  "strengths": ["point 1", "point 2"],
  "weaknesses": ["point 1", "point 2"],
  "improved_answer": "one short paragraph"
}}

Rules:
- Output only JSON
- No markdown
- No code fences
- improved_answer must be one single string
- strengths must have exactly 2 items
- weaknesses must have exactly 2 items
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return result

In [36]:
question = "What is the difference between GET, POST, PUT, and DELETE in REST APIs?"

answer = """
GET is used to retrieve data from the server.
POST is used to create a new resource.
PUT is used to update an existing resource, usually replacing it.
DELETE is used to remove a resource from the server.
For example, in a student API, GET /students fetches students,
POST /students creates a new student, PUT /students/1 updates student 1,
and DELETE /students/1 removes student 1.
"""

print(evaluate_answer_json(question, answer))

```json
{
  "technical_accuracy": 3,
  "clarity": 4,
  "depth": 3,
  "strengths": ["correct explanation of HTTP methods for REST APIs"],
  "weaknesses": ["redundant examples could be more helpful"],
  "improved_answer": "In a RESTful API, GET retrieves data, POST creates or updates resources, PUT replaces or updates resources, and DELETE removes resources."
}
```


In [37]:
import re
import json

def clean_and_parse_json(raw_output):
    cleaned = raw_output.strip()

    # remove ```json and ```
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        data = json.loads(cleaned)
        return data
    except json.JSONDecodeError as e:
        print("JSON parsing failed:")
        print(e)
        print("\nCleaned output was:\n", cleaned)
        return None

In [38]:
raw_output = """
{
  "technical_accuracy": 3,
  "clarity": 4,
  "depth": 3,
  "strengths": ["correct explanation of HTTP methods for REST APIs"],
  "weaknesses": ["redundant examples could be more helpful"],
  "improved_answer": "In a RESTful API, GET retrieves data, POST creates or updates resources, PUT replaces or updates resources, and DELETE removes resources."
}
"""

parsed = clean_and_parse_json(raw_output)
print(parsed)

{'technical_accuracy': 3, 'clarity': 4, 'depth': 3, 'strengths': ['correct explanation of HTTP methods for REST APIs'], 'weaknesses': ['redundant examples could be more helpful'], 'improved_answer': 'In a RESTful API, GET retrieves data, POST creates or updates resources, PUT replaces or updates resources, and DELETE removes resources.'}


In [39]:
import re
import json

# -----------------------------
# 1) Question generator
# -----------------------------
def generate_question(role, level, topic, difficulty):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional mock interviewer. "
                "Generate exactly one interview question. "
                "Do not give the answer."
            )
        },
        {
            "role": "user",
            "content": f"""
Role: {role}
Level: {level}
Topic: {topic}
Difficulty: {difficulty}

Generate one realistic interview question only.
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return result


# -----------------------------
# 2) Evaluator
# -----------------------------
def evaluate_answer_json(question, answer):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert technical interviewer. "
                "Return ONLY valid JSON. "
                "Do not use markdown. "
                "Do not use code fences. "
                "The improved_answer must be a single string."
            )
        },
        {
            "role": "user",
            "content": f"""
Question: {question}

Candidate Answer:
{answer}

Evaluate the answer on:
- technical_accuracy (0 to 5)
- clarity (0 to 5)
- depth (0 to 5)

Return exactly this JSON structure:
{{
  "technical_accuracy": 0,
  "clarity": 0,
  "depth": 0,
  "strengths": ["point 1", "point 2"],
  "weaknesses": ["point 1", "point 2"],
  "improved_answer": "one short paragraph"
}}
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return result


# -----------------------------
# 3) JSON cleaner/parser
# -----------------------------
def clean_and_parse_json(raw_output):
    cleaned = raw_output.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        data = json.loads(cleaned)
        return data
    except json.JSONDecodeError as e:
        print("JSON parsing failed:")
        print(e)
        print("\nCleaned output was:\n", cleaned)
        return None


# -----------------------------
# 4) Adaptive logic
# -----------------------------
def choose_adaptive_action(scores):
    technical = scores["technical_accuracy"]
    clarity = scores["clarity"]
    depth = scores["depth"]

    avg_score = (technical + clarity + depth) / 3

    if technical <= 2:
        return {
            "next_difficulty": "Easy",
            "focus": "same topic",
            "reason": "technical weakness detected"
        }
    elif clarity <= 2:
        return {
            "next_difficulty": "Medium",
            "focus": "explanation-style question",
            "reason": "clarity weakness detected"
        }
    elif depth <= 2:
        return {
            "next_difficulty": "Medium",
            "focus": "deeper follow-up question",
            "reason": "depth weakness detected"
        }
    elif avg_score >= 4:
        return {
            "next_difficulty": "Hard",
            "focus": "next related topic",
            "reason": "strong overall performance"
        }
    else:
        return {
            "next_difficulty": "Medium",
            "focus": "same topic with moderate challenge",
            "reason": "average performance"
        }


# -----------------------------
# 5) One interview round
# -----------------------------
def run_interview_round(role, level, topic, difficulty, answer):
    question = generate_question(role, level, topic, difficulty)
    raw_eval = evaluate_answer_json(question, answer)
    parsed_eval = clean_and_parse_json(raw_eval)

    if parsed_eval is None:
        return {
            "question": question,
            "raw_evaluation": raw_eval,
            "parsed_evaluation": None,
            "adaptive_decision": None
        }

    decision = choose_adaptive_action(parsed_eval)

    return {
        "question": question,
        "raw_evaluation": raw_eval,
        "parsed_evaluation": parsed_eval,
        "adaptive_decision": decision
    }

In [40]:
result = run_interview_round(
    role="Backend Developer",
    level="Junior",
    topic="REST API",
    difficulty="Easy",
    answer="""
GET is used to retrieve data from the server.
POST is used to create a new resource.
PUT is used to update an existing resource, usually replacing it.
DELETE is used to remove a resource from the server.
For example, in a student API, GET /students fetches students,
POST /students creates a new student, PUT /students/1 updates student 1,
and DELETE /students/1 removes student 1.
"""
)

print("QUESTION:\n", result["question"])
print("\nPARSED EVALUATION:\n", result["parsed_evaluation"])
print("\nADAPTIVE DECISION:\n", result["adaptive_decision"])

QUESTION:
 Please describe your experience with creating and implementing a simple RESTful API using popular frameworks such as Flask or Django. How did you handle authentication, rate limiting, and error handling in your implementation?

PARSED EVALUATION:
 {'technical_accuracy': 0, 'clarity': 0, 'depth': 0, 'strengths': [], 'weaknesses': [], 'improved_answer': 'The GET method retrieves data, POST for creation, PUT for updating, and DELETE for removal.'}

ADAPTIVE DECISION:
 {'next_difficulty': 'Easy', 'focus': 'same topic', 'reason': 'technical weakness detected'}


In [29]:
print(evaluate_answer(question, answer))

Technical Accuracy: 4/5
Clarity: 4/5
Depth: 3/5

Strengths:
- Correctly identifies the main differences between HTTP methods in RESTful APIs.

Weaknesses:
- The explanation could be more detailed or specific about each method's purpose and examples.
- The examples provided for GET, POST, PUT, and DELETE are somewhat generic and might not cover all possible scenarios.


In [25]:
def choose_next_difficulty(technical, clarity, depth, current_difficulty):
    avg_score = (technical + clarity + depth) / 3

    if avg_score >= 4:
        return "Hard"
    elif avg_score >= 2.5:
        return "Medium"
    else:
        return "Easy"

In [26]:
print(choose_next_difficulty(4, 4, 3, "Medium"))
print(choose_next_difficulty(2, 2, 2, "Medium"))
print(choose_next_difficulty(5, 4, 5, "Medium"))

Medium
Easy
Hard


In [27]:
def choose_adaptive_action(technical, clarity, depth):
    if technical < 3:
        return {
            "next_difficulty": "Easy",
            "focus": "same topic",
            "reason": "technical weakness detected"
        }
    elif clarity < 3:
        return {
            "next_difficulty": "Medium",
            "focus": "explanation-style question",
            "reason": "clarity weakness detected"
        }
    elif depth < 3:
        return {
            "next_difficulty": "Medium",
            "focus": "follow-up conceptual question",
            "reason": "depth weakness detected"
        }
    else:
        return {
            "next_difficulty": "Hard",
            "focus": "next related topic",
            "reason": "strong performance"
        }

In [23]:
question = "What is the difference between GET, POST, PUT, and DELETE in REST APIs?"

answer = """
GET is used to retrieve data from the server.
POST is used to create a new resource.
PUT is used to update an existing resource, usually replacing it.
DELETE is used to remove a resource from the server.
For example, in a student API, GET /students fetches students,
POST /students creates a new student, PUT /students/1 updates student 1,
and DELETE /students/1 removes student 1.
"""

print(evaluate_answer(question, answer))

Technical Accuracy: 4/5
Clarity: 4/5
Depth: 3/5

Strengths:
- Correctly identifies the main differences between HTTP methods in RESTful APIs.

Weaknesses:
- The explanation could be more detailed or specific about each method. For instance, explaining that GET typically returns data without side effects, while POST can modify state on the server. Similarly, for PUT and DELETE, discussing how they handle resources and their implications on the client-side state.


In [16]:
question = "What is the difference between GET, POST, PUT, and DELETE in REST APIs?"

answer = "GET fetches data, POST creates data, PUT updates data, and DELETE removes data."

print(evaluate_answer(question, answer))


You are an expert technical interviewer.

Evaluate the candidate's answer based on the question.

Question:
What is the difference between GET, POST, PUT, and DELETE in REST APIs?

Candidate Answer:
GET fetches data, POST creates data, PUT updates data, and DELETE removes data.

Score the answer using this rubric (0-5):

Technical Accuracy
Clarity
Depth of Explanation

Return the result in this format:

Technical Accuracy: X/5
Clarity: X/5
Depth: X/5

Strengths:
- bullet points

Weaknesses:
- bullet points

Improved Answer:
Provide a short improved version of the answer.
- Bullet point for Technical Accuracy
- Bullet point for Clarity
- Bullet point for Depth
- Bullet point for Strengths
- Bullet point for Weaknesses ```Technical Accuracy: 4/5
Clarity: 3/5
Depth: 2/5
Strengths:
- Concise yet informative
- Directly answers the question without unnecessary details
- Easy to understand
- Lack of technical jargon makes it accessible
- Clear examples could be provided for better understand

In [17]:
temperature=0.3

In [18]:
question = "What is the difference between GET, POST, PUT, and DELETE in REST APIs?"

answer = """
GET is used to retrieve data from the server.
POST is used to create a new resource.
PUT is used to update an existing resource, usually replacing it.
DELETE is used to remove a resource from the server.
For example, in a student API, GET /students fetches students,
POST /students creates a new student, PUT /students/1 updates student 1,
and DELETE /students/1 removes student 1.
"""

print(evaluate_answer(question, answer))


You are an expert technical interviewer.

Evaluate the candidate's answer based on the question.

Question:
What is the difference between GET, POST, PUT, and DELETE in REST APIs?

Candidate Answer:

GET is used to retrieve data from the server.
POST is used to create a new resource.
PUT is used to update an existing resource, usually replacing it.
DELETE is used to remove a resource from the server.
For example, in a student API, GET /students fetches students,
POST /students creates a new student, PUT /students/1 updates student 1,
and DELETE /students/1 removes student 1.


Score the answer using this rubric (0-5):

Technical Accuracy
Clarity
Depth of Explanation

Return the result in this format:

Technical Accuracy: X/5
Clarity: X/5
Depth: X/5

Strengths:
- bullet points

Weaknesses:
- bullet points

Improved Answer:
Provide a short improved version of the answer.
Sure! Let me evaluate your candidate’s answer based on the provided questions:

**Technical Accuracy:** 4/5
**Clarity

In [21]:
def evaluate_answer_json(question, answer):
    prompt = f"""
You are an expert technical interviewer.

Evaluate the candidate's answer.

Question: {question}
Candidate Answer: {answer}

Return only valid JSON in this exact structure:

{{
  "technical_accuracy": 0,
  "clarity": 0,
  "depth": 0,
  "strengths": ["", ""],
  "weaknesses": ["", ""],
  "improved_answer": ""
}}

Rules:
- Scores must be integers from 0 to 5
- Output only JSON
- No markdown
- No code block
- No extra text
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=220,
        temperature=0.2,
        top_p=0.9,
        do_sample=True
    )

    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    result = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text.strip()

    return result